# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [4]:


import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Get the PRICE_DATA path
price_data_path = os.getenv("PRICE_DATA")
print("PRICE_DATA path:", price_data_path)



PRICE_DATA path: ../../05_src/data/prices/


In [5]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [6]:
import os
from glob import glob

# Write your code below.

from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Get the PRICE_DATA path
price_data_path = os.getenv("PRICE_DATA")
print("PRICE_DATA path:", price_data_path)

# Find all parquet files in the PRICE_DATA directory
parquet_files = glob(os.path.join(price_data_path, "*.parquet"))

# Print the list of files found
print("Parquet files found:", parquet_files)




PRICE_DATA path: ../../05_src/data/prices/
Parquet files found: []


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [7]:
# Write your code below.
print("Found parquet files:", parquet_files)

import os
from glob import glob
import dask.dataframe as dd
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
price_data_path = os.getenv("PRICE_DATA")

# Get all .parquet file paths
parquet_files = glob(os.path.join(price_data_path, "*.parquet"))

# Create list to store individual DataFrames
ddf_list = []

for file in parquet_files:
    try:
        df = dd.read_parquet(file)

        # Ensure expected columns exist
        if set(['Close', 'Adj_Close', 'High', 'Low']).issubset(df.columns):
            # Sort by Date if available
            if 'Date' in df.columns:
                df = df.sort_values('Date')

            # Add lag columns
            df['Close_lag_1'] = df['Close'].shift(1)
            df['Adj_Close_lag_1'] = df['Adj_Close'].shift(1)

            # Add returns
            df['returns'] = (df['Close'] / df['Close_lag_1']) - 1

            # Add high-low range
            df['hi_lo_range'] = df['High'] - df['Low']

            # Append to list
            ddf_list.append(df)
        else:
            print(f"Skipping {file}: missing required columns")
    except Exception as e:
        print(f"Failed to process {file}: {e}")

# Combine all dataframes
if ddf_list:
    dd_feat = dd.concat(ddf_list)
    print("Successfully created dd_feat")
else:
    print("No valid dataframes were processed.")


Found parquet files: []
No valid dataframes were processed.


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [8]:
# Write your code below.

import os
from glob import glob
import dask.dataframe as dd
from dotenv import load_dotenv

load_dotenv()
price_data_path = os.getenv("PRICE_DATA")
print("PRICE_DATA path:", price_data_path)

parquet_files = glob(os.path.join(price_data_path, "*.parquet"))
print(f"Found {len(parquet_files)} parquet files:", parquet_files)

ddf_list = []

for file in parquet_files:
    try:
        df = dd.read_parquet(file)
        print(f"Columns in {file}:", df.columns.tolist())

        required_cols = ['Close', 'Adj_Close', 'High', 'Low']
        if all(col in df.columns for col in required_cols):
            if 'Date' in df.columns:
                df = df.sort_values('Date')

            df['Close_lag_1'] = df['Close'].shift(1)
            df['Adj_Close_lag_1'] = df['Adj_Close'].shift(1)
            df['returns'] = (df['Close'] / df['Close_lag_1']) - 1
            df['hi_lo_range'] = df['High'] - df['Low']
            ddf_list.append(df)
        else:
            print(f"Skipping {file} — missing required columns")
    except Exception as e:
        print(f"Error loading {file}: {e}")

print(f"Number of DataFrames ready to concat: {len(ddf_list)}")

if ddf_list:
    dd_feat = dd.concat(ddf_list)
    print("dd_feat created successfully")
else:
    print("No valid DataFrames to concatenate — dd_feat not created")

try:
    df = dd_feat.compute()
    if 'Date' in df.columns:
        df = df.sort_values('Date')
    df['returns_ma_10'] = df['returns'].rolling(10).mean()
    print(df.head())
except NameError:
    print("dd_feat is not defined. Please check previous steps.")




PRICE_DATA path: ../../05_src/data/prices/
Found 0 parquet files: []
Number of DataFrames ready to concat: 0
No valid DataFrames to concatenate — dd_feat not created
dd_feat is not defined. Please check previous steps.


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
Not always. But in this case, we used pandas because it was simple and easy for small data.
+ Would it have been better to do it in Dask? Why?
Yes, if the data is very big. Dask can handle large data that doesn’t fit in memory. It also makes the process faster by using multiple cores. But Dask code is a bit more complex than pandas.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.